Data: Open University Learning Analytics Dataset (OULAD)

https://www.kaggle.com/datasets/anlgrbz/student-demographics-online-education-dataoulad/data?select=studentAssessment.csv

Research Question: 
Does having higher length of course interaction (from vle table) lead to higher score (student assessment)?

In [1]:
import pandas as pd

# Step 1: Data Exploration

## Read in Data

In [2]:
courses = pd.read_csv('Data/courses.csv')
assessments = pd.read_csv('Data/assessments.csv')
stu_assess = pd.read_csv('Data/studentAssessment.csv')
regist = pd.read_csv('Data/studentRegistration.csv')
vle = pd.read_csv('Data/vle.csv')

print(courses.head())
print(assessments.head())
print(stu_assess.head())
print(regist.head())
print(vle.head())

  code_module code_presentation  module_presentation_length
0         AAA             2013J                         268
1         AAA             2014J                         269
2         BBB             2013J                         268
3         BBB             2014J                         262
4         BBB             2013B                         240
  code_module code_presentation  id_assessment assessment_type   date  weight
0         AAA             2013J           1752             TMA   19.0    10.0
1         AAA             2013J           1753             TMA   54.0    20.0
2         AAA             2013J           1754             TMA  117.0    20.0
3         AAA             2013J           1755             TMA  166.0    20.0
4         AAA             2013J           1756             TMA  215.0    30.0
   id_assessment  id_student  date_submitted  is_banked  score
0           1752       11391              18          0   78.0
1           1752       28400              22  

## Analyze the Resource Type for Each Class

* Only include the course that have one activity type throughout the course by keeping rows where the week_from and week_to columns are both missing.

In [3]:
vle_cleaned = vle[(vle['week_from'].isnull()) & (vle['week_to'].isnull())]
#  Class-year-activity level
vle_cleaned_group_act = vle_cleaned.groupby(['code_module', 'code_presentation','activity_type']).size().reset_index(name = 'cnt_act_type')
# Class-year level
vle_cleaned_group = vle_cleaned_group_act.groupby(['code_module', 'code_presentation']).agg(
    tot_cnt = ('cnt_act_type', 'sum'),
    most_freq_act = ('cnt_act_type', 'max')
).reset_index()
vle_comb = vle_cleaned_group_act.merge(vle_cleaned_group, on = ['code_module', 'code_presentation'])
# calculate the percentage of each activity amoung each class
vle_comb['act_percent'] = round((vle_comb['cnt_act_type']*100 / vle_comb['tot_cnt']), 2)
print(vle_comb.describe())


       cnt_act_type     tot_cnt  most_freq_act  act_percent
count    229.000000  229.000000     229.000000   229.000000
mean      22.895197  245.344978     115.489083     9.607336
std       42.720095  125.644824      59.346116    15.654614
min        1.000000   83.000000      32.000000     0.210000
25%        2.000000  173.000000      77.000000     0.980000
50%        6.000000  218.000000      95.000000     2.850000
75%       19.000000  315.000000     182.000000     8.650000
max      236.000000  483.000000     236.000000    81.280000


* find the most frequently used activity type for each class

In [4]:
# find the most frequently used activity type for each class
most_freq_act = vle_comb[vle_comb['cnt_act_type'] == vle_comb['most_freq_act']]
print(most_freq_act)

    code_module code_presentation activity_type  cnt_act_type  tot_cnt  \
6           AAA             2013J      resource            95      208   
15          AAA             2014J      resource            93      199   
24          BBB             2013B      resource           236      315   
34          BBB             2013J      resource           196      245   
44          BBB             2014B      resource           191      235   
55          BBB             2014J      resource            96      173   
64          CCC             2014B      resource            78      191   
73          CCC             2014J      resource            85      218   
84          DDD             2013B      resource           182      428   
95          DDD             2013J       subpage           194      462   
105         DDD             2014B       subpage           193      452   
114         DDD             2014J      resource           169      364   
125         EEE             2013J     

**Conclusion of the analysis**: Most of the materials among different classes are classified as 'resource', which consist of 33%-81% of all the available activities for each class.

## Analyze the Student

* Only include the students that did not drop out


In [5]:
undrop_stu = regist[regist['date_unregistration'].isnull()]
all_crs_stu = undrop_stu.merge(courses, how = 'left', on = ['code_module','code_presentation'])


* Find the assessment of each of the classes that each student took

In [6]:
course_assess= all_crs_stu.merge(assessments, how = 'left', on = ['code_module','code_presentation'])
stu_course_score = course_assess.merge(stu_assess, how = 'left', on = ['id_assessment', 'id_student'])

* Find the final score at the course-student level

In [7]:
stu_course_score = stu_course_score.drop(['date_unregistration'], axis = 1)
stu_course_score = stu_course_score.dropna()

* Make sure the assessment_type only consist of TMA, CMA and Exam

In [9]:
stu_course_score[['assessment_type']].drop_duplicates()

,assessment_type
0,TMA
3732,CMA
85677,Exam


* Separate the Final Exam, TMA and CMA

In [12]:
# Final
final_exam = stu_course_score[stu_course_score['assessment_type'] =='Exam']

# TMA
tma = stu_course_score[stu_course_score['assessment_type'] =='TMA']

# CMA
cma = stu_course_score[stu_course_score['assessment_type'] =='CMA']

In [17]:
print('Tutor Marked Assessment (TMA)')
print(tma.describe())

print('\n\nComputer Marked Assessment (CMA)')
print(cma.describe())

Tutor Marked Assessment (TMA)
         id_student  date_registration  module_presentation_length  \
count  8.953600e+04       89536.000000                89536.000000   
mean   7.095677e+05         -66.699015                  256.048584   
std    5.586499e+05          46.909243                   13.121811   
min    6.516000e+03        -310.000000                  234.000000   
25%    5.045960e+05         -95.000000                  241.000000   
50%    5.872970e+05         -53.000000                  262.000000   
75%    6.424442e+05         -29.000000                  268.000000   
max    2.698588e+06         124.000000                  269.000000   

       id_assessment          date        weight  date_submitted  \
count   89536.000000  89536.000000  89536.000000    89536.000000   
mean    25164.538041     98.571245     16.903335       97.341706   
std      9081.128121     59.760061      8.344443       61.125869   
min      1752.000000     12.000000      0.000000      -11.000000   

**Conclusion of the analysis**:
* There is more TMA than CMA
* Looking at the mean and the percentile distribution (25th, 50th and 75th), students perform better at CMA type of quiz than TMA type of quiz

In [27]:
stu_course_score['weighted_score'] = stu_course_score['score'] * stu_course_score['weight']*0.01

stu_course_lvl = stu_course_score.groupby(['id_student', 'code_module', 'code_presentation']).agg(
    final_score =('weighted_score', 'sum'),
    tot_weight = ('weight', 'sum')
).reset_index()
print(stu_course_lvl[stu_course_lvl['tot_weight'].isin([100])])
# print(stu_course_lvl[stu_course_lvl['tot_weight']!= 100])
# print(stu_course_lvl[stu_course_lvl['tot_weight']== 200])


       id_student code_module code_presentation  final_score  tot_weight
0            6516         AAA             2014J        63.50       100.0
1           11391         AAA             2013J        82.40       100.0
3           23698         CCC             2014J        69.97       100.0
4           23798         BBB             2013J        89.24       100.0
8           24734         AAA             2014J        47.50       100.0
...           ...         ...               ...          ...         ...
21160     2698125         FFF             2013J        68.00       100.0
21162     2698257         AAA             2013J        69.40       100.0
21163     2698535         EEE             2013J        53.44       100.0
21164     2698577         BBB             2014J        55.80       100.0
21165     2698588         BBB             2014J        92.40       100.0

[10977 rows x 5 columns]


In [28]:
stu_course_lvl.describe()

,id_student,final_score,tot_weight
count,2.116600e+04,21166.000000,21166.000000
mean,7.066363e+05,61.891283,85.182084
std,5.507277e+05,38.693220,51.073866
min,6.516000e+03,0.000000,0.000000
25%,5.073440e+05,34.772500,57.000000
50%,5.895555e+05,68.250000,100.000000
75%,6.425910e+05,84.227500,100.000000
max,2.698588e+06,197.900000,200.000000


In [32]:
questionable = stu_course_lvl[stu_course_lvl['tot_weight'].isin([200])]
questionable

,id_student,code_module,code_presentation,final_score,tot_weight
6,24213,DDD,2014B,136.400,200.0
24,28046,DDD,2013J,89.900,200.0
33,29411,DDD,2013J,137.650,200.0
49,31173,DDD,2013J,77.125,200.0
90,40184,DDD,2013J,193.000,200.0
...,...,...,...,...,...
21144,2693243,DDD,2013B,170.580,200.0
21149,2694886,DDD,2014B,138.875,200.0
21151,2694933,DDD,2013B,149.970,200.0
21153,2695608,DDD,2013J,152.650,200.0


In [34]:
ques = stu_course_score.merge(questionable, how = 'right', on = ['id_student', 'code_module', 'code_presentation'])

In [35]:
ques = ques[(ques['id_student'] == 24213) & (ques['code_module'] =='DDD') & (ques['code_presentation'] == '2014B')]
ques

,code_module,code_presentation,id_student,date_registration,module_presentation_length,id_assessment,assessment_type,date,weight,date_submitted,is_banked,score,weighted_score,final_score,tot_weight
0,DDD,2014B,24213,-54.0,241,25355,TMA,25.0,10.0,25.0,0.0,78.0,7.800,136.4,200.0
1,DDD,2014B,24213,-54.0,241,25356,TMA,53.0,12.5,54.0,0.0,91.0,11.375,136.4,200.0
2,DDD,2014B,24213,-54.0,241,25357,TMA,74.0,17.5,78.0,0.0,87.0,15.225,136.4,200.0
3,DDD,2014B,24213,-54.0,241,25358,TMA,116.0,20.0,123.0,0.0,67.0,13.400,136.4,200.0
4,DDD,2014B,24213,-54.0,241,25359,TMA,158.0,20.0,163.0,0.0,89.0,17.800,136.4,200.0
5,DDD,2014B,24213,-54.0,241,25360,TMA,200.0,20.0,201.0,0.0,64.0,12.800,136.4,200.0
6,DDD,2014B,24213,-54.0,241,25361,Exam,241.0,100.0,236.0,0.0,58.0,58.000,136.4,200.0


In [23]:
print(stu_course_score[stu_course_score['is_banked']!= 0])

       code_module code_presentation  module_presentation_length  \
1641           AAA             2014J                         269   
1685           AAA             2014J                         269   
1690           AAA             2014J                         269   
1693           AAA             2014J                         269   
1708           AAA             2014J                         269   
...            ...               ...                         ...   
167452         GGG             2014J                         269   
168095         GGG             2014J                         269   
168106         GGG             2014J                         269   
168848         GGG             2014J                         269   
168858         GGG             2014J                         269   

        id_assessment assessment_type   date  weight  id_student  \
1641             1758             TMA   19.0    10.0      603861   
1685             1758             TMA   19.0   

In [28]:
print('Hello')

Hello
